[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/es/lab4/lab4.ipynb)
# Práctica 4: Redes neuronales con Regularización
## El sobreajuste (overfitting)
El problema del sobreajuste (*overfitting*) consiste en que la solución aprendida se ajusta muy bien a los datos de entrenamiento, pero no generaliza adecuadamente ante la aparición de nuevos datos.

Para observar el sobreajuste en un problema, revisitaremos el problema de clasificación con datos del Titanic que hemos utilizado en los anteriores laboratorios. En primer lugar, carga los datos tal como se hacía en los anteriores notebooks. Asegúrate de tener una partición de entrenamiento con el 50\% de los datos y una de test con el 25\% y una de validación con el 25\% restante. Utiliza la función [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) de `sklearn` sobre el `ndarray` de datos en dos veces sucesivas para conseguir las particiones. Después pasa los conjuntos resultantes a `torch` y envuélvelos en un `Dataloader` como se hacía en el laboratorio anterior.

**IMPORTANTE:** La separación de conjuntos debe hacerse sobre los datos sin preprocesar. Todo preprocesado se debe hacer sobre cada conjunto por separado, pero utilizando solo información del conjunto de entrenamiento. **Revisa la carga de datos de Titanic** para que la estandarización (paso 4 de la función de carga de datos) se realice después de la partición. La función devolverá los datos sin escalar y, tras hacer la separación en train/test/val, utilizaremos un único `StandardScaler` con el que haremos `fit_transform` sobre train y `transform` sobre test y val. Así evitaremos que se filtre ningún tipo de información de test/val al entrenamiento.

In [ ]:
# TODO - Carga el dataset Titanic en tres dataloaders: train/test/val
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(1234567)
torch.manual_seed(1234567)

def load_titanic_raw():
    """Carga y codifica el dataset SIN escalar. Devuelve X (sin escalar) e y."""
    df = sns.load_dataset('titanic')
    cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
    df = df[cols].copy()
    df = df.dropna(subset=['age', 'embarked', 'fare'])

    y = df['survived'].to_numpy().astype(np.float32)
    X = df.drop(columns=['survived'])

    categorical_cols = ['pclass', 'sex', 'embarked', 'alone']
    X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

    # Devolvemos SIN escalar (el escalado se hace después de la partición)
    X_np = X_encoded.to_numpy().astype(np.float32)
    y_np = y.reshape(-1, 1).astype(np.float32)
    return X_np, y_np

X, y = load_titanic_raw()

# Partición 50% train / 25% val / 25% test
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.5, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

# Escalado: fit SOLO con train
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Convertimos a tensores
X_train = torch.from_numpy(X_train.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
X_val   = torch.from_numpy(X_val.astype(np.float32))
y_val   = torch.from_numpy(y_val.astype(np.float32))
X_test  = torch.from_numpy(X_test.astype(np.float32))
y_test  = torch.from_numpy(y_test.astype(np.float32))

# DataLoaders
batch_size = 32
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val),     batch_size=batch_size)
test_loader  = DataLoader(TensorDataset(X_test, y_test),   batch_size=batch_size)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Una vez hayas cargado los datos, vamos a diseñar dos funciones para entrenar y evaluar un modelo. Las funciones deben tener esta forma:
- `def measure_model(model:nn.Module, loss_fn:nn.modules.loss._Loss, data_loader:data.DataLoader) -> tuple[float, float]:`
    - Devuelve una tupla con la pérdida y la accuracy en el conjunto pasado como parámetro. Asegúrate de poner el modelo en modo `eval`.
- `def train_model(model:nn.Module,
                optimizer:optim.Optimizer,
                train_loader:data.DataLoader,
                loss_fn:nn.modules.loss._Loss,
                num_epochs:int,
                val_loader:data.DataLoader,
                patience:None|int=None) -> tuple[tuple[list[int], list[int]]:`
    - Entrena el modelo con los datos de `train_loader` y devuelve una tupla con el histórico de losses en `train_loader` y en `val_loader`. Asegúrate de poner el modelo en modo `train`.

In [ ]:
def measure_model(model:nn.Module, loss_fn:nn.modules.loss._Loss, data_loader:data.DataLoader) -> tuple[float, float]:
    #TODO Completa la función
    # Devuelve (pérdida_media, accuracy) en el conjunto dado.
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in data_loader:
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            total_loss += loss.item() * x.size(0)

            # Para BCELoss, la salida ya está en [0,1]; umbral 0.5
            preds = (y_pred >= 0.5).float()
            total_correct += (preds == y).sum().item()
            total_samples += x.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

def train_model(model:nn.Module,
                optimizer:optim.Optimizer,
                train_loader:data.DataLoader,
                loss_fn:nn.modules.loss._Loss,
                num_epochs:int,
                val_loader:data.DataLoader=None,
                ) -> tuple[tuple[list[int], list[int]]:

    #TODO Completa la función
    # Entrena el modelo y devuelve (train_losses, val_losses).
    train_losses = []
    val_losses = []

    best_val_loss = float('inf')
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(num_epochs):
        # --- Entrenamiento ---
        model.train()
        running_loss = 0.0
        n_samples = 0
        for x, y in train_loader:
            optimizer.zero_grad()
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            n_samples += x.size(0)

         train_loss = running_loss / n_samples
        train_losses.append(train_loss)

        # --- Validación ---
        if val_loader is not None:
            val_loss, _ = measure_model(model, loss_fn, val_loader)
            val_losses.append(val_loss)

            # Early stopping
            if patience is not None:
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    epochs_without_improvement = 0
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                else:
                    epochs_without_improvement += 1
                    if epochs_without_improvement >= patience:
                        print(f"Early stopping en epoch {epoch+1}")
                        if best_state is not None:
                            model.load_state_dict(best_state)
                        break
    return train_losses, val_losses

Para este primer apartado, diseña una red neuronal totalmente conectada con tres capas ocultas de 20, 10 y 5 neuronas respectivamente.

In [ ]:
#TODO Define la subclase de nn.Module con la arquitectura descrita
class TitanicNet(nn.Module):
    def __init__(self, input_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 20),
            nn.ReLU(),
            nn.Linear(20, 10),
            nn.ReLU(),
            nn.Linear(10, 5),
            nn.ReLU(),
            nn.Linear(5, 1),
            nn.Sigmoid()   # Usamos BCELoss, por eso sigmoide al final
        )

    def forward(self, x):
        return self.net(x)

Ahora entrena el modelo utilizando el optimizador `Adam` y la función de coste `BCELoss`. Tras entrenar, calcula la accuracy en test y muestra con `matplotlib` la curva de entrenamiento.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import pyplot

model = TitanicNet(input_dim=X_train.shape[1])
# print(model)

# Definimos la función de pérdida y el optimizador
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 700

train_losses, val_losses = train_model(
    model, optimizer, train_loader, loss_fn, num_epochs, val_loader)

pyplot.plot(train_losses, label="train")
pyplot.plot(val_losses, label="val")
pyplot.legend()
pyplot.show()

test_loss, test_accuracy = measure_model(model, loss_fn, test_loader)
print(f'Test loss:{test_loss:.4f}\tTest accuracy:{test_accuracy:.2f}')

En la curva de entrenamiento deberías ser capaz de identificar que hay sobreentrenamiento porque la curva de validación llega a un punto que deja de mejorar y comienza a empeorar.

*NOTA: Puedes exagerar el efecto de sobreajuste tomando menos datos de entrenamiento, lo cual facilita su memorización.*

# Regularización
Una vez diagnosticado el sobreajuste, es hora de probar diferentes técnicas que intenten reducir la varianza, sin incrementar demasiado el sesgo y, con ello, el modelo generaliza mejor. Las técnicas de regularización que vamos a ver en este laboratorio son:
1. *Early stopping*. Detiene el entrenamiento de la red cuando aumenta el error.
1. Penalización basada	en	la	norma	de	los	parámetros (tanto norma L1 como L2).
1. *Dropout*. Ampliamente utilizada en aprendizaje profundo, "desactiva" algunas neuronas para evitar el sobreajuste.

## Parte 1. Early Stopping
Hacer parada temprana (*early stopping*) consiste en monitorizar el rendimiento en un pequeño subconjunto del conjunto de entrenamiento y parar el aprendizaje cuando este rendimiento decaiga.

Para llevarlo a cabo, añade a la función de `train_model` un parámetro opcional llamado `patience` que indique durantos cuántos pasos puede empeorar el la pérdida en validación sin que se pare el entrenamiento. Cuando esté presente el valor de `patience` el entrenamiento se tiene que parar atendiendo a él.

Una vez tengas la función, vuelve a declarar el modelo y optimizador, entrena y revisa las curvas y el rendimiento en test.

In [ ]:
# TODO - Instancia un modelo y un optimizador, haz el entrenamiento con early stopping y muestra las curvas y medidas
model_es = TitanicNet(input_dim=X_train.shape[1])
loss_fn = nn.BCELoss()
optimizer_es = optim.Adam(model_es.parameters(), lr=0.001)

train_losses_es, val_losses_es = train_model(
    model_es, optimizer_es, train_loader, loss_fn,
    num_epochs=700, val_loader=val_loader, patience=20
)

pyplot.plot(train_losses_es, label="train")
pyplot.plot(val_losses_es, label="val")
pyplot.legend()
pyplot.xlabel("Epoch")
pyplot.ylabel("Loss")
pyplot.title("Early Stopping (patience=20)")
pyplot.show()

test_loss_es, test_accuracy_es = measure_model(model_es, loss_fn, test_loader)
print(f'Test loss:{test_loss_es:.4f}\tTest accuracy:{test_accuracy_es:.2f}')

# Parte 2. Penalización basada en norma de parámetros
Otra alternativa para regularizar consiste en añadir a la función de pérdida una componente que penalice los valores altos en los parámetros. En `torch` le podemos indicar al optimizador que incluya una penalización L2 en los pesos indicando el parámetro `weight_decay` al instanciarlo. Prueba distintos valores (suele ser buena política probar cambios en factores de 10 alrededor del 0.001) y comprueba su efecto en el rendimiento. No utilices *early stopping* en el entrenamiento.

**¿Qué diferencia observas en la curva de entrenamiento con respecto a utilizar *early stopping*?**

In [ ]:
# TODO - Instancia un modelo y un optimizador con weight decay, haz el entrenamiento y muestra las curvas y medidas
for wd in [0.0001, 0.001, 0.01, 0.1]:
    model_wd = TitanicNet(input_dim=X_train.shape[1])
    optimizer_wd = optim.Adam(model_wd.parameters(), lr=0.001, weight_decay=wd)

    train_losses_wd, val_losses_wd = train_model(
        model_wd, optimizer_wd, train_loader, loss_fn, num_epochs=700, val_loader=val_loader
    )

    test_loss_wd, test_acc_wd = measure_model(model_wd, loss_fn, test_loader)
    print(f"weight_decay={wd:.4f}  Test loss={test_loss_wd:.4f}  Test acc={test_acc_wd:.2f}")

    pyplot.plot(train_losses_wd, label=f"train (wd={wd})")
    pyplot.plot(val_losses_wd,   label=f"val (wd={wd})")

pyplot.legend()
pyplot.xlabel("Epoch")
pyplot.ylabel("Loss")
pyplot.title("Penalización L2 (weight decay)")
pyplot.show()

# Parte 3. Dropout
Por último, la tercera alternativa que exploraremos para regularizar es añadir capas `nn.Dropout` a nuestra arquitectura. Estas capas apagan durante el entrenamiento una fracción (que debemos indicar como parámetro al instanciar la capa) de las neuronas que reciben a la entrada, impidiendo al modelo apoyarse siempre en las mismas entradas para hacer sus predicciones, forzándolo así a aprender representaciones más generales y, por tanto, mejorando su generalización.

Crea una nueva arquitectura de red con estas capas y entrénala (sin usar ningún otro mecanismo de regularización). Prueba a situar las capas `nn.Dropout` en distintos sitios y a aplicar distintos valores del parámetro `p` que indica la probabilidad de que se apague cada neurona.

In [ ]:

# TODO - Declara la nueva clase, haz que tu modelo sea una instancia de ella, instancia el optimizador, haz el entrenamiento y muestra las curvas y medidas.
class TitanicNetDropout(nn.Module):
    def __init__(self, input_dim=10, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 20),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(20, 10),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(10, 5),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(5, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


for p in [0.1, 0.3, 0.5]:
    model_do = TitanicNetDropout(input_dim=X_train.shape[1], p=p)
    optimizer_do = optim.Adam(model_do.parameters(), lr=0.001)

    train_losses_do, val_losses_do = train_model(
        model_do, optimizer_do, train_loader, loss_fn, num_epochs=700, val_loader=val_loader
    )

    test_loss_do, test_acc_do = measure_model(model_do, loss_fn, test_loader)
    print(f"Dropout p={p}  Test loss={test_loss_do:.4f}  Test acc={test_acc_do:.2f}")

    pyplot.plot(train_losses_do, label=f"train (p={p})")
    pyplot.plot(val_losses_do,   label=f"val (p={p})")

pyplot.legend()
pyplot.xlabel("Epoch")
pyplot.ylabel("Loss")
pyplot.title("Dropout")
pyplot.show()